In [1]:
import os

import logging
import time
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import csv
import pandas as pd 
import numpy as np 

from eutils import EutilsNCBIError, EutilsRequestError
from metapub import PubMedFetcher, pubmedcentral
from datetime import datetime

from Functions import DataRetrieval as DR

#API_KEY
from Reference_files.keys import API_KEY as API_KEY

In [ ]:
API_KEY = "70faf5cc42501a814dcc4bdb1862acaf3909"

In [ ]:
API_KEY = API_KEY

In [2]:
# Initialize logger
prefix = "test"+str(datetime.now()).split()[0]
file_handler = logging.FileHandler(f"{prefix}_Examples.log", mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

# From Query to pubmedCentral full text

### the following pipeline starts with a file with the query which is in the folder Reference files to PMC full text
#### pubmed central 

In [3]:
# Initialize PubMed fetcher
fetcher = PubMedFetcher()

In [ ]:
@DR.retry_on_communication_error
def fetch_pmcid(pmid):
    try:
        pmc = pubmedcentral.get_pmcid_for_otherid(pmid)
        return pmc
    except (CommunicationError, ConnectionError) as e:
        logging.error(f"Error: API request failed for {pmid}: {e}")
        return None
    except Exception as e:
        logging.error(f"Unexpected error for {pmid}: {e}")
        return None

def get_pmcid_for_otherid(pmid_clean_list):
    PMCIDs = []
    with ThreadPoolExecutor(max_workers=10) as executor:  # Adjust max_workers based on needs
        future_to_pmid = {executor.submit(fetch_pmcid, pmid): pmid for pmid in pmid_clean_list}
        for future in as_completed(future_to_pmid):
            pmid = future_to_pmid[future]
            try:
                pmc = future.result()
                PMCIDs.append(pmc)
            except Exception as e:
                logging.error(f"Error processing PMID {pmid}: {e}")
                PMCIDs.append(None)
    return PMCIDs

def filter_oa_database(oa_file_list, pmc_id_list):
    """
    Filters based on the csv database list of PMCs that are available for full_text mining.

    Parameters:
    oa_file_list (csv): Filename of the CSV containing the OA file list.
    pmc_id_list (list): Filename of the list containing the PMC IDs.
    """
    # Read CSV file
    oa_file_list_df = pd.read_csv(oa_file_list)
    # Filter oa_database based on PMC ID list
    filtered_oa_database = oa_file_list_df[oa_file_list_df["Accession ID"].isin(pmc_id_list)]
    return filtered_oa_database

In [5]:
query_file = "./Reference_files/query.txt"
query = DR.read_query_from_file(query_file)

In [6]:
# Step 2: Fetch PMIDs over a specified period
start_date = "2000-01-01"
stop_date = None  # Will default to the current date if None
pmid_array = DR.fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)

2025-03-28 20:28:57 LAPTOP-S8N3C7A8 root[23868] INFO Total PMIDs fetched: 271


In [7]:
pmc_id_list = get_pmcid_for_otherid(pmid_array)

In [ ]:
oa_file_list = "Coa_file_list.csv"
oa_pmcs = filter_oa_database(oa_file_list, pmc_id_list)

In [11]:
def main():
    # Step 1: Read the query from a file
    query_file = "./Reference_files/query.txt" 
    query = DR.read_query_from_file(query_file)
    if not query:
        logging.error("Query reading failed. Exiting.")
        return
    # Step 2: Fetch PMIDs over a specified period
    start_date = "2000-01-01"
    stop_date = None  # Will default to the current date if None
    pmid_array = DR.fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)
    if pmid_array.size == 0:
        logging.error("No PMIDs fetched. Exiting.")
        return
    # Step 3: Retrieve PMCIDs for the fetched PMIDs
    pmc_id_list = get_pmcid_for_otherid(pmid_array)
    # Step 5: Filter the OA database using the retrieved PMCIDs
    oa_file_list = "oa_file_list.csv"  # OA file list CSV filename
    oa_pmcs = filter_oa_database(oa_file_list, pmc_id_list)

    logging.info("OA database filtering completed.")
    return oa_pmcs

# Call the main function to execute the workflow
if __name__ == "__main__":
    main()


2025-03-28 23:29:27 LAPTOP-S8N3C7A8 root[23868] INFO Total PMIDs fetched: 271


FileNotFoundError: [Errno 2] No such file or directory: 'oa_file_list.csv'

In [ ]:
!curl -s "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC10759277/unicode" > "PMC10759277.json"


In [ ]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${url}" > "./Full_text_jsons/${PMCID}.json"
done < full_text_pmc.txt



In [ ]:
!powershell -Command "mkdir -p ./Full_text_jsons; Get-Content full_text_pmc.txt | ForEach-Object { Invoke-RestMethod -Uri ('https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/' + $_ + '/unicode') -OutFile ('./Full_text_jsons/' + $_ + '.json') }"

In [ ]:

pmids = ["12345678", "23456789"]  # Example PMIDs
try1 = DR.fetch_articles_to_dataframe(pmids)

# Publisher Metadata

# Paper Metadata